In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


## load data

In [2]:
train_data = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')
train_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
test_data = pd.read_csv('/kaggle/input/competitions/titanic/test.csv')
test_data.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [4]:
gender_submission_data = pd.read_csv('/kaggle/input/competitions/titanic/gender_submission.csv')
gender_submission_data.head()

,PassengerId,Survived
0,892,0
1,893,1
2,894,0
3,895,0
4,896,1


## analysis

### expolore

In [5]:
women = train_data.loc[train_data.Sex == "female"]["Survived"]
rate_women = sum(women) / len(women)
print(f"{rate_women} of women who survived")

0.7420382165605095 of women who survived


In [6]:
men = train_data.loc[train_data.Sex == "male"]["Survived"]
rate_men = sum(men) / len(men)
print(f"{rate_men} of men who survived")

0.18890814558058924 of men who survived


In [7]:
train_data.isna().values.any()
train_data["Age"].isna().values.any()

np.True_

In [8]:
train_data[train_data["Age"].isna()]

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
17,18,1,2,"Williams, Mr. Charles Eugene",male,NaN,0,0,244373,13.0000,NaN,S
19,20,1,3,"Masselmani, Mrs. Fatima",female,NaN,0,0,2649,7.2250,NaN,C
26,27,0,3,"Emir, Mr. Farred Chehab",male,NaN,0,0,2631,7.2250,NaN,C
28,29,1,3,"O'Dwyer, Miss. Ellen ""Nellie""",female,NaN,0,0,330959,7.8792,NaN,Q
...,...,...,...,...,...,...,...,...,...,...,...,...
859,860,0,3,"Razi, Mr. Raihed",male,NaN,0,0,2629,7.2292,NaN,C
863,864,0,3,"Sage, Miss. Dorothy Edith ""Dolly""",female,NaN,8,2,CA. 2343,69.5500,NaN,S
868,869,0,3,"van Melkebeke, Mr. Philemon",male,NaN,0,0,345777,9.5000,NaN,S
878,879,0,3,"Laleff, Mr. Kristo",male,NaN,0,0,349217,7.8958,NaN,S


In [9]:
age_nan_count = train_data["Age"].isna().sum()
print(age_nan_count / len(train_data["Age"]) * 100)

19.865319865319865


In [10]:
train_data.index[train_data["Age"].isna()]

Index([  5,  17,  19,  26,  28,  29,  31,  32,  36,  42,
       ...
       832, 837, 839, 846, 849, 859, 863, 868, 878, 888],
      dtype='int64', length=177)

In [11]:
train_data.index[train_data["Fare"].isna()]

Index([], dtype='int64')

In [12]:
nan_fare_idx = test_data.index[test_data["Fare"].isna()]
print(nan_fare_idx[0])

152


In [13]:
test_data.iloc[nan_fare_idx[0]]

PassengerId                  1044
Pclass                          3
Name           Storey, Mr. Thomas
Sex                          male
Age                          60.5
SibSp                           0
Parch                           0
Ticket                       3701
Fare                          NaN
Cabin                         NaN
Embarked                        S
Name: 152, dtype: object

## preprocess

### Calculate the median age from titles
1. 敬称ごとに中央値を算出
2. 欠損値の名前を見て敬称を抽出
3. 敬称ごとの中央値を割り当てる

In [14]:
def extract_titles(data):
    return data["Name"].str.split(", ").str[1].str.split(" ").str[0]

In [15]:
# 1. 敬称ごとに中央値を算出
def median_by_title(data):
    titles = extract_titles(data)
    return data.groupby(titles)["Age"].median().to_dict()

In [16]:
titles = extract_titles(train_data)
print(titles)

0        Mr.
1       Mrs.
2      Miss.
3       Mrs.
4        Mr.
       ...  
886     Rev.
887    Miss.
888    Miss.
889      Mr.
890      Mr.
Name: Name, Length: 891, dtype: object


In [17]:
nan_names = train_data[train_data["Age"].isna()]["Name"]

In [18]:
def impute_missing_val(data):
    data = data.copy()
    age_by_title = median_by_title(data)
    overall_median = data["Age"].median()

    # 2. 欠損値の名前を見て敬称を抽出
    titles = extract_titles(data)
    
    # 3. 敬称ごとの中央値を割り当てる
    filler = titles.map(age_by_title).fillna(overall_median)
    data["Age"] = data["Age"].fillna(filler)
    return data

In [19]:
train_data = impute_missing_val(train_data)
test_data = impute_missing_val(test_data)
train_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [20]:
train_data.index[train_data["Age"].isna()]

Index([], dtype='int64')

### test_dataのfare欠損値を埋める
1. train_dataのpclass別fare辞書を作成
2. test_dataの欠損値に適用

## submission

### ramdom forest

In [21]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_squared_error as MSE
from sklearn.metrics import r2_score

In [22]:
def train(n_estimators, max_depth):
    model = RandomForestClassifier(
        n_estimators = n_estimators, 
        max_depth = max_depth,
        random_state = 0
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    return model, y_pred

#### 目的変数と説明変数の分離

In [23]:
y_train = train_data["Survived"]
features = ["Pclass", "Sex", "SibSp", "Parch", "Age"]

#### カテゴリ変数rをone-hot化
sk-learnは文字列を扱えないため、数値に変換

SexはmaleとfemaleなのでSex_maleとSex_femaleに分離する

In [24]:
X_train = pd.get_dummies(train_data[features])
X_test = pd.get_dummies(test_data[features])

#### modelの設定

In [25]:
model, y_pred = train(100, 5)

- n_estimators=100: 決定木を100本作り、多数決で予測する
- max_depth=5: 各木の深さを5に制限し、過学習を抑える
- random_state=1: 乱数固定。実行ごとに結果が変わらないようにする

## evaluate

In [26]:
from sklearn.model_selection import cross_val_score, RepeatedStratifiedKFold

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=0)
scores = cross_val_score(model, X_train, y_train, cv=cv, n_jobs=-1)
print(f"{scores.mean():.4f} ± {scores.std():.4f}  (n={len(scores)})")

0.8285 ± 0.0254  (n=50)


## output

In [27]:
output = pd.DataFrame({
    "PassengerId": test_data.PassengerId,
    "Survived": y_pred
})
output.to_csv("submission.csv", index=False)
print("Save csv")

Save csv
